In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

integration_dir = Path.cwd().resolve()
if str(integration_dir) not in sys.path:
    sys.path.insert(0, str(integration_dir))

import integration_runtime as ir

print("Integration dispatcher ready.")
print("Routing table:")
print("  .log                           -> BNN CAN Vehicle Classifier")
print("  .csv                           -> SVM Lane Classifier")
print("  .wav                           -> MLP Siren Classifier")
print("  .png / .jpg / .jpeg / ...      -> CNN Traffic Sign Classifier")


In [ ]:

import os
import json
import subprocess
import sys as _sys

# ── Inline routing + execution (fully self-contained, does not rely on ir.*) ──
_EXT_MAP = {
    ".log":  "bnn",
    ".csv":  "svm",
    ".wav":  "mlp",
    ".png":  "cnn", ".jpg": "cnn", ".jpeg": "cnn",
    ".bmp":  "cnn", ".tif": "cnn", ".tiff": "cnn",
}
_LABELS = {
    "bnn": "BNN CAN Vehicle Classifier",
    "cnn": "CNN Traffic Sign Classifier",
    "svm": "SVM Lane Classifier",
    "mlp": "MLP Siren Classifier",
}
_NOTEBOOKS = {
    "bnn": "bnn_inference_pynq.ipynb",
    "cnn": "cnn_accelerator_script.ipynb",
    "svm": "svm_accelerator_script.ipynb",
    "mlp": "mlp_inference_pynq.ipynb",
}

def _infer_model(filename):
    ext = Path(filename).suffix.strip().lower()
    if ext not in _EXT_MAP:
        raise ValueError(f"Unknown extension '{ext}'. Supported: {sorted(_EXT_MAP)}")
    return _EXT_MAP[ext]

def _extract_nb_output(nb_path):
    """Read all stream/text outputs from an executed notebook JSON."""
    try:
        with open(nb_path) as f:
            nb = json.load(f)
        lines = []
        for cell in nb.get("cells", []):
            for out in cell.get("outputs", []):
                otype = out.get("output_type", "")
                if otype == "stream":
                    text = out.get("text", "")
                elif otype in ("execute_result", "display_data"):
                    text = "".join(out.get("data", {}).get("text/plain", []))
                elif otype == "error":
                    text = f"[ERROR] {out.get('ename', '')}: {out.get('evalue', '')}"
                else:
                    text = ""
                if isinstance(text, list):
                    text = "".join(text)
                if text.strip():
                    lines.append(text.rstrip())
        return "\n".join(lines)
    except Exception as e:
        return f"(could not read notebook output: {e})"

def _execute_notebook(model_key, input_path):
    """Run the model notebook via nbconvert, injecting env vars for path resolution."""
    nb_src = integration_dir / "model_notebooks" / _NOTEBOOKS[model_key]
    if not nb_src.exists():
        raise FileNotFoundError(f"Notebook not found: {nb_src}")
    out_nb = Path("/tmp") / f"executed_{model_key}.ipynb"
    env = os.environ.copy()
    env["INTEGRATION_BASE_DIR"] = str(integration_dir)
    env["ROUTED_INPUT_PATH"]    = str(input_path)
    cmd = [
        _sys.executable, "-m", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--ExecutePreprocessor.timeout=300",
        "--output", str(out_nb),
        str(nb_src),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
    return {
        "executed_notebook": str(out_nb),
        "cell_output": _extract_nb_output(out_nb) if out_nb.exists() else "",
        "stderr": proc.stderr,
        "returncode": proc.returncode,
    }

# ── File path input ────────────────────────────────────────────────────────────
path_input = widgets.Text(
    value="",
    placeholder="/home/xilinx/jupyter_notebooks/Integration/data/test.wav",
    description="File path:",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "80px"},
)

run_button = widgets.Button(
    description="Run Routed Notebook",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)

output = widgets.Output()


def _run(_):
    with output:
        output.clear_output()

        file_path = path_input.value.strip().strip("'\"")
        if not file_path:
            print("Enter the full path to your test file in the box above, then click the button.")
            return

        input_path = Path(file_path)
        if not input_path.exists():
            print(f"File not found: {input_path}")
            print("Tip: run  !ls /home/xilinx/jupyter_notebooks/Integration/  to list files.")
            return

        try:
            model_key = _infer_model(input_path.name)
            print(f"File      : {input_path}")
            print(f"Extension : {input_path.suffix}")
            print(f"Model     : {_LABELS[model_key]}")
            print("\nExecuting notebook ...\n")

            result = _execute_notebook(model_key, str(input_path))

            rc = result.get("returncode", -1)
            print("Done." if rc == 0 else f"Warning: nbconvert exited with code {rc}")
            print(f"Executed  : {result['executed_notebook']}")

            cell_out = result.get("cell_output", "").strip()
            if cell_out:
                print("\n--- Result ---")
                print(cell_out)

            # Show nbconvert stderr only on failure
            if rc != 0:
                stderr = result.get("stderr", "").strip()
                if stderr:
                    print("\n--- nbconvert log ---")
                    print(stderr)

        except Exception as exc:
            import traceback
            print(f"Error: {exc}")
            traceback.print_exc()


run_button.on_click(_run)
display(path_input, run_button, output)
